# 🛠️ Ingestão de Dados Brutos — Camada Bronze (SISCAN / 10ª CRS)

Este notebook é responsável pela **etapa de ingestão e estruturação inicial** dos dados públicos sobre exames citopatológicos do colo do útero (SISCAN) na **Camada Bronze** da Arquitetura Medallion.

---

## 1. Descompactação e Organização dos Arquivos Fonte

**Objetivo:** Extrair a estrutura de diretórios do arquivo `dataset.zip` e salvá-la no diretório de destino do Volume no Unity Catalog (`siscan_files_bronze/dataset/`), garantindo a preservação da hierarquia original sem duplicação de pastas.

* **Fonte:** `/Volumes/tcc-analise-siscan-10crs/bronze/siscan_files_bronze/dataset.zip`
* **Destino:** `/Volumes/tcc-analise-siscan-10crs/bronze/siscan_files_bronze/dataset/`

In [0]:
import zipfile
import shutil
import os

# Caminho do volume e do ficheiro zip
volume_path = "/Volumes/tcc-analise-siscan-10crs/bronze/siscan_files_bronze"
zip_path = f"{volume_path}/dataset.zip"
target_dataset_dir = f"{volume_path}/dataset"
temp_extract_dir = f"{volume_path}/temp_extraicao"

# 1. Garante que a pasta 'dataset' exista
os.makedirs(target_dataset_dir, exist_ok=True)

# 2. Extrai o zip numa pasta temporária
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(temp_extract_dir)

# 3. Se dentro do zip já existia uma pasta 'dataset', move o conteúdo dela diretamente;
# caso contrário, move o conteúdo da extração temporária
nested_dir = os.path.join(temp_extract_dir, "dataset")
source_dir = nested_dir if os.path.exists(nested_dir) else temp_extract_dir

for item in os.listdir(source_dir):
    src = os.path.join(source_dir, item)
    dst = os.path.join(target_dataset_dir, item)
    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.move(src, dst)
    else:
        shutil.move(src, dst)

# 4. Remove a pasta temporária de trabalho
if os.path.exists(temp_extract_dir):
    shutil.rmtree(temp_extract_dir)

print("Dados extraídos e organizados com sucesso dentro de '/dataset'!")

## 2. Ingestão dos CSVs Brutos para Tabelas Delta (Bronze)

**Objetivo:** Ler os arquivos CSV brutos extraídos do SISCAN/DATASUS e persisti-los como tabelas em formato **Delta Lake** no catálogo `tcc-analise-siscan-10crs.bronze`.

### Diretrizes de Engenharia de Dados Aplicadas na Camada Bronze:
1. **Fidelidade à Fonte (*Raw Data*):** Os dados são gravados no estado original sem limpezas, renomeações de colunas ou filtros de negócios.
2. **Suporte a Caracteres Especiais (`Column Mapping`):** Uso do parâmetro `delta.columnMapping.mode = "name"` para preservar nomes de colunas originais do SISCAN (com acentos, espaços e pontos) sem falhas no esquema.
3. **Rastreabilidade e Auditoria:** Adição automatizada dos metadados `_ingestion_time` (timestamp da carga) e `_file_source` (caminho do arquivo de origem via `_metadata.file_path`).

---
### 📊 Mapeamento de Tabelas Geradas:
* **`bronze.exames_geral_raw`:** Matriz histórica de quantidade de exames (Q1 e Q2).
* **`bronze.faixa_etaria_raw`:** Distribuição de exames por faixas etárias (Q3).
* **`bronze.adequabilidade_raw`:** Classificação da adequabilidade das amostras (Q4).
* **`bronze.resultados_raw`:** Categorias dos resultados citopatológicos alterados (Q5).

In [0]:
from pyspark.sql import functions as F

# Base de leitura dos ficheiros brutos
PATH_RAW = "/Volumes/tcc-analise-siscan-10crs/bronze/siscan_files_bronze/dataset"

# 1. Ingestão de Exames Gerais (Q1 e Q2)
df_geral = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"{PATH_RAW}/q1_q2_exames_geral_2020_2025/*.csv")
    .withColumn("_ingestion_time", F.current_timestamp())
    .withColumn("_file_source", F.col("_metadata.file_path"))
)

(
    df_geral.write.format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .saveAsTable("`tcc-analise-siscan-10crs`.bronze.exames_geral_raw")
)

# 2. Ingestão de Faixa Etária (Q3)
df_faixa_etaria = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"{PATH_RAW}/Q3_Faixa_Etaria/*.csv")
    .withColumn("_ingestion_time", F.current_timestamp())
    .withColumn("_file_source", F.col("_metadata.file_path"))
)

(
    df_faixa_etaria.write.format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .saveAsTable("`tcc-analise-siscan-10crs`.bronze.faixa_etaria_raw")
)

# 3. Ingestão de Adequabilidade (Q4)
df_adequabilidade = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"{PATH_RAW}/Q4_Adequabilidade/*.csv")
    .withColumn("_ingestion_time", F.current_timestamp())
    .withColumn("_file_source", F.col("_metadata.file_path"))
)

(
    df_adequabilidade.write.format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .saveAsTable("`tcc-analise-siscan-10crs`.bronze.adequabilidade_raw")
)

# 4. Ingestão de Resultados Citopatológicos (Q5)
df_resultados = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ";")
    .load(f"{PATH_RAW}/Q5_Resultados_Citopatologicos/*.csv")
    .withColumn("_ingestion_time", F.current_timestamp())
    .withColumn("_file_source", F.col("_metadata.file_path"))
)

(
    df_resultados.write.format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .saveAsTable("`tcc-analise-siscan-10crs`.bronze.resultados_raw")
)

## 3. Validação da Carga e Auditoria de Registros

**Objetivo:** Verificar a persistência física dos dados, realizar contagem de linhas e inspecionar a amostragem das tabelas Delta recém-criadas na Camada Bronze.

In [0]:
# 2. Verificar a quantidade total de registros em cada tabela
print("Exames Geral:", spark.table("`tcc-analise-siscan-10crs`.bronze.exames_geral_raw").count())
print("Faixa Etária:", spark.table("`tcc-analise-siscan-10crs`.bronze.faixa_etaria_raw").count())
print("Adequabilidade:", spark.table("`tcc-analise-siscan-10crs`.bronze.adequabilidade_raw").count())
print("Resultados:", spark.table("`tcc-analise-siscan-10crs`.bronze.resultados_raw").count())

### 3.1. Inspeção Visual da Tabela Delta (`exames_geral_raw`)

**Objetivo:** Exibir a amostragem dos dados brutos persistidos na tabela `exames_geral_raw` em formato de planilha interativa.

* **Metadados:** Permite verificar a presença das colunas técnicas `_ingestion_time` e `_file_source` adicionadas no pipeline de ingestão[cite: 1, 15].
* **Estrutura Matricial:** Apresenta as 73 colunas de meses/anos originais do SISCAN/TABNET associadas aos 12 municípios da 10ª CRS[cite: 1, 6, 15].

In [0]:
# 1. Visualizar o conteúdo em formato de planilha interativa
display(spark.table("`tcc-analise-siscan-10crs`.bronze.exames_geral_raw"))